# Title

## 1: Imports e Configurações

In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm
from scipy.optimize import brentq
from torch.utils.data import DataLoader, Subset

project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
# Imports do seu projeto
from src.config import PATHS, DATA_CONFIG, MODEL_CONFIG, VIZ_CONFIG, TRAINING_CONFIG
from src.model import DeepHestonHybrid
from src.data_loader import criar_dataset_hibrido
from src.visualization import Visualizer
from src.logger import setup_logger

# Configuração de Logger
logger = setup_logger(name='Notebook_Avaliacao', log_dir='resultados')

# Detectar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de avaliação: {device}")

[INFO] Log file criado: resultados\training_20260217_223602.log
Dispositivo de avaliação: cuda


## 2: Carregamento do Modelo e Dados

In [2]:
import os
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, Subset, TensorDataset
from src.config import PATHS, DATA_CONFIG, MODEL_CONFIG, VIZ_CONFIG
from src.data_loader import carregar_taxa_juros, criar_dataset_hibrido
from src.model import DeepHestonHybrid
from src.visualization import Visualizer

# Configurar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# 1. Carregar Dados Atuais (Novos)
# ==============================================================================

# Passo A: Carregar a taxa de juros
print("Carregando taxa de juros...")
df_juros = carregar_taxa_juros(PATHS['selic_data'])

if df_juros is None:
    raise ValueError("Erro ao carregar arquivo da Selic. Verifique o caminho em config.py")

# Passo B: Criar o dataset com os dados atuais
print("Criando dataset híbrido (Dados Atuais)...")
full_dataset, current_data_stats = criar_dataset_hibrido(
    PATHS['raw_data'], # Caminho da pasta de opções
    df_juros,          # DataFrame de juros carregado acima
    seq_length=DATA_CONFIG.get('sequence_length', 30)
)

# ==============================================================================
# 2. Resgatar Configuração do Treinamento Original (CORREÇÃO DE ERRO)
# ==============================================================================
# Para evitar "RuntimeError: size mismatch", precisamos saber quantos ativos 
# existiam quando o modelo foi salvo.

stats_path = os.path.join(PATHS['model_save_dir'], 'data_stats.json')
weights_path = os.path.join(PATHS['model_save_dir'], 'best_model_weights.pth')

if not os.path.exists(stats_path):
    # Tenta caminho alternativo se não achar no principal
    stats_path = os.path.join(PATHS['results_dir'], 'modelo_final', 'data_stats.json')
    weights_path = os.path.join(PATHS['results_dir'], 'modelo_final', 'best_model_weights.pth')

if os.path.exists(stats_path):
    print(f"Carregando estatísticas do treinamento original: {stats_path}")
    with open(stats_path, 'r') as f:
        saved_data_stats = json.load(f)
else:
    raise FileNotFoundError(f"Arquivo data_stats.json não encontrado. Necessário para recuperar a arquitetura do modelo.")

# ==============================================================================
# 3. Filtragem e Ajuste de IDs (Compatibilidade com Debug)
# ==============================================================================
target_ticker = 'PETR'

# Normalização de chaves (remove espaços em branco e garante maiúsculas)
current_asset_map = {k.strip().upper(): v for k, v in current_data_stats.get('asset_map', {}).items()}
saved_asset_map = {k.strip().upper(): v for k, v in saved_data_stats.get('asset_map', {}).items()}

print("\n--- Diagnóstico de Ativos ---")
print(f"Buscando por: '{target_ticker}'")
print(f"Ativos no dataset atual ({len(current_asset_map)}): {list(current_asset_map.keys())}")
print(f"Ativos no modelo salvo ({len(saved_asset_map)}): {list(saved_asset_map.keys())}")

if target_ticker in current_asset_map and target_ticker in saved_asset_map:
    # IDs podem ter mudado entre o treino e agora
    current_id = current_asset_map[target_ticker]
    trained_id = saved_asset_map[target_ticker]
    
    print(f"\nSucesso! Filtrando ativo {target_ticker}.")
    print(f"Mapeamento de ID: {current_id} (Atual) -> {trained_id} (Treinado)")
    
    # Pega os índices da PETR4 no dataset atual
    # O tensor de IDs é o último elemento da tupla no TensorDataset (índice 5)
    all_ids = full_dataset.tensors[5]
    indices_petr4 = (all_ids == current_id).nonzero(as_tuple=True)[0]
    
    if len(indices_petr4) == 0:
        raise ValueError(f"O ativo {target_ticker} existe no mapa, mas não foram encontrados dados com ID {current_id} no tensor.")

    # Define split de validação (últimos 20%)
    test_split = int(len(indices_petr4) * 0.2)
    # Garante pelo menos 1 amostra se o dataset for pequeno
    test_split = max(1, test_split) 
    val_indices = indices_petr4[-test_split:]
    
    print(f"Total de amostras PETR4: {len(indices_petr4)}. Usando {len(val_indices)} para validação.")

    # --- CRIAÇÃO DE DATASET CORRIGIDO ---
    # Extrai os tensores apenas da validação
    X_seq = full_dataset.tensors[0][val_indices]
    X_phy = full_dataset.tensors[1][val_indices]
    y = full_dataset.tensors[2][val_indices]
    X_time = full_dataset.tensors[3][val_indices]
    weights = full_dataset.tensors[4][val_indices]
    
    # AQUI ESTÁ O TRUQUE: Criamos um tensor de IDs com o valor que o MODELO espera (trained_id)
    # e não o valor que o dataset novo gerou (current_id).
    ids_corrected = torch.full((len(val_indices),), trained_id, dtype=torch.long)
    
    val_dataset = TensorDataset(X_seq, X_phy, y, X_time, weights, ids_corrected)
    
else:
    # Diagnóstico de falha
    missing_in = []
    if target_ticker not in current_asset_map: missing_in.append("Dataset Atual")
    if target_ticker not in saved_asset_map: missing_in.append("Modelo Salvo")
    
    raise ValueError(f"Ativo {target_ticker} não encontrado em: {', '.join(missing_in)}.")

# Criar DataLoader
val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False)
print(f"Dataloader de validação pronto: {len(val_dataset)} amostras (IDs ajustados).")

# ==============================================================================
# 4. Carregar Modelo e Executar
# ==============================================================================
# Instancia o modelo usando saved_data_stats para garantir num_assets=14 (exemplo)
NUM_ASSETS_CHECKPOINT = 14  # Valor descoberto através da mensagem de erro

# Cria uma cópia da configuração para não estragar a original
forced_data_stats = saved_data_stats.copy()

# Se o mapa salvo tiver tamanho diferente, criamos um mapa dummy com o tamanho certo
# apenas para satisfazer a inicialização da camada de embedding.
if len(forced_data_stats.get('asset_map', {})) != NUM_ASSETS_CHECKPOINT:
    print(f"AVISO: data_stats.json tem {len(forced_data_stats.get('asset_map', {}))} ativos, "
          f"mas o checkpoint exige {NUM_ASSETS_CHECKPOINT}. Forçando arquitetura correta.")
    
    # Recria um mapa dummy com chaves genéricas para garantir len() == 14
    dummy_map = {f'ASSET_{i}': i for i in range(NUM_ASSETS_CHECKPOINT)}
    
    # Mantém o ID do alvo (PETR) correto se possível, ou mapeia para um ID válido
    target_ticker_clean = target_ticker.strip().upper()
    
    # Se o ID treinado (trained_id) for menor que 14, podemos usar ele.
    if 'trained_id' in locals() and trained_id < NUM_ASSETS_CHECKPOINT:
        # Tudo certo, o ID treinado é válido na nova arquitetura
        pass
    else:
        # Se o ID treinado era, digamos, 18, e agora só temos 14 slots, temos um problema grave.
        # Mas assumindo que o erro diz [14, 4], é provável que trained_id esteja entre 0 e 13.
        pass

    forced_data_stats['asset_map'] = dummy_map

# Instancia o modelo com a configuração forçada
model = DeepHestonHybrid(MODEL_CONFIG, forced_data_stats).to(device)

if os.path.exists(weights_path):
    print(f"Carregando pesos de: {weights_path}")
    # strict=False pode ajudar se houver pequenas diferenças em buffers não treináveis,
    # mas para pesos de camadas (como embedding) as dimensões TEM que bater.
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"Modelo carregado com sucesso!")
else:
    raise FileNotFoundError(f"Pesos não encontrados em {weights_path}")

# Inicializar Visualizador
# Usamos forced_data_stats para manter consistência
history_path = os.path.join(PATHS['results_dir'], 'training_history.csv')
viz = Visualizer(model, history_path, val_loader, forced_data_stats, VIZ_CONFIG)

# Gerar Cache de Inferência
try:
    df_results = viz.run_inference()
    print(f"Inferência concluída. DataFrame shape: {df_results.shape}")
    # display(df_results.head()) # Descomente se estiver no Jupyter
except Exception as e:
    print(f"Erro na inferência: {e}")
    import traceback
    traceback.print_exc()

Carregando taxa de juros...
Criando dataset híbrido (Dados Atuais)...
[INFO] Lendo arquivos de opções em: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\dados\brutos
[INFO] IBOV carregado com sucesso: 443085 registros, coluna de preço: acao_close_ajustado
[INFO] IBOV carregado com 576 datas
[INFO] Intervalos de dados carregados, com 2332315 registos.
[INFO] Data de início 2019-12-09 00:00:00 
[INFO] Data do fim 2023-12-28 00:00:00
[INFO] Ativos encontrados: ['ABEV' 'B3SA' 'BBAS' 'BBDC' 'BOVA' 'CSNA' 'GGBR' 'ITUB' 'LREN' 'MGLU'
 'PETR' 'SUZB' 'VALE' 'WEGE']
[INFO] Asset Map: {'ABEV': 0, 'B3SA': 1, 'BBAS': 2, 'BBDC': 3, 'BOVA': 4, 'CSNA': 5, 'GGBR': 6, 'ITUB': 7, 'LREN': 8, 'MGLU': 9, 'PETR': 10, 'SUZB': 11, 'VALE': 12, 'WEGE': 13}
[INFO] Gerando sequências...
[INFO] Dataset Final: 2296104 amostras.
[INFO] Target (P/K) - Mean: 0.1093, Std: 5.3850
[INFO] Moneyness Raw - Mean: 1.0681, Std: 7.4735
[INFO] S_norm - Mean: 0.0000, Std: 1.0000
[INFO] K_norm - Mean: -0.0000, Std: 1.0000
[INFO] Asse

## 3: Diagnóstico de Treinamento (Losses)

In [3]:
# Plota convergência das perdas
fig_loss = viz.plot_loss_convergence_hybrid() # Certifique-se que este método existe no seu visualization.py atualizado
if fig_loss: fig_loss.show()

# Plota detecção de Overfitting
fig_overfit = viz.plot_overfitting_detection()
if fig_overfit: fig_overfit.show()

[INFO] ✓ Loss history salvo: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\loss_convergence.html


## 4: Análise de Erro Financeiro

In [4]:
# Tabela de Métricas
rmse = np.sqrt(np.mean(df_results['Error']**2))
mae = np.mean(df_results['Abs_Error'])
mape = np.mean(np.abs(df_results['Error'] / df_results['Price_Real'])) * 100
r2 = 1 - (np.sum(df_results['Error']**2) / np.sum((df_results['Price_Real'] - df_results['Price_Real'].mean())**2))

print(f"=== Métricas de Desempenho ===")
print(f"RMSE: R$ {rmse:.4f}")
print(f"MAE:  R$ {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"R²:   {r2:.4f}")

# Plot Scatter (Real vs Previsto)
fig_scatter = px.scatter(
    df_results.sample(n=min(2000, len(df_results))), 
    x='Price_Real', y='Price_Pred', 
    color='Moneyness', title="Aderência: Preço de Mercado vs PINN",
    trendline="ols"
)
fig_scatter.add_shape(type="line", x0=0, y0=0, x1=df_results['Price_Real'].max(), y1=df_results['Price_Real'].max(), line=dict(color="Red", dash="dash"))
fig_scatter.show()

=== Métricas de Desempenho ===
RMSE: R$ 2.0890
MAE:  R$ 0.1603
MAPE: 8247.03%
R²:   0.8415


## 5: O "Santo Graal" - Curva de Volatilidade Implícita (Smile)

In [5]:
# --- Função Helper: Inversão Black-Scholes (Newton-Raphson) ---
from scipy.optimize import brentq

def implied_vol_robust(price, S, K, T, r, type_='call'):
    """Cálculo robusto de Volatilidade Implícita (evita overflow)."""
    def bs_diff(sigma):
        # Implementação segura de BS
        try:
            d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
            d2 = d1 - sigma * np.sqrt(T)
            if type_ == 'call':
                bs_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
            else:
                bs_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
            return bs_price - price
        except:
            return -price # Retorna erro se der overflow

    try:
        # Busca raiz entre 1% e 500% de volatilidade
        return brentq(bs_diff, 0.01, 5.0)
    except:
        return np.nan # Falha graciosa

# --- Aplicação no Dataset ---
print("Calculando Volatilidade Implícita (Isso pode demorar um pouco)...")

# Filtrar fatia representativa para o plot (ex: T entre 20 e 60 dias)
# Usamos uma amostra para ser rápido
sample_df = df_results[(df_results['T'] > 0.08) & (df_results['T'] < 0.25)].copy()
sample_df = sample_df.sample(n=min(500, len(sample_df))) # 500 pontos para o gráfico ficar limpo

# Calcular IV para Mercado e para PINN
# Assumimos taxa livre de risco r=0.10 (ajuste conforme seu dataset se tiver a coluna 'r')
r_val = 0.10 
sample_df['IV_Market'] = sample_df.apply(lambda x: implied_vol_robust(x['Price_Real'], x['S'], x['K'], x['T'], r_val), axis=1)
sample_df['IV_PINN'] = sample_df.apply(lambda x: implied_vol_robust(x['Price_Pred'], x['S'], x['K'], x['T'], r_val), axis=1)

# Plotar o Smile
fig_smile = go.Figure()
fig_smile.add_trace(go.Scatter(x=sample_df['Moneyness'], y=sample_df['IV_Market'], mode='markers', name='Mercado (Real)', marker=dict(color='green', size=6)))
fig_smile.add_trace(go.Scatter(x=sample_df['Moneyness'], y=sample_df['IV_PINN'], mode='markers', name='PINN (Previsto)', marker=dict(color='blue', symbol='x', size=6)))

fig_smile.update_layout(
    title="Validação do Smile de Volatilidade (Implied Vol)",
    xaxis_title="Moneyness (S/K)",
    yaxis_title="Volatilidade Implícita",
    hovermode="closest"
)
fig_smile.show()

Calculando Volatilidade Implícita (Isso pode demorar um pouco)...


## 6: Validação Física via Monte Carlo

In [6]:
# --- Função Helper: Monte Carlo Heston ---
def heston_mc_simulation(S0, K, T, r, kappa, theta, xi, rho, v0, num_paths=5000, num_steps=50):
    dt = T / num_steps
    prices = np.zeros(num_paths)
    vt = np.zeros(num_paths) + v0
    St = np.zeros(num_paths) + S0
    
    for t in range(num_steps):
        z1 = np.random.normal(size=num_paths)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.normal(size=num_paths)
        
        # Euler-Maruyama com truncamento para variância positiva
        vt = np.maximum(vt + kappa * (theta - vt) * dt + xi * np.sqrt(vt * dt) * z2, 0)
        St = St * np.exp((r - 0.5 * vt) * dt + np.sqrt(vt * dt) * z1)
        
    payoffs = np.maximum(St - K, 0)
    return np.mean(payoffs) * np.exp(-r * T)

# --- Execução ---
# Seleciona 1 caso aleatório do dataset de teste
case = df_results.sample(1).iloc[0]

print(f"--- Validação Física (Monte Carlo) ---")
print(f"Parâmetros Inferidos pela LSTM para este cenário:")
print(f"Kappa: {case['Heston_kappa']:.4f}, Theta: {case['Heston_theta']:.4f}, Xi: {case['Heston_xi']:.4f}, Rho: {case['Heston_rho']:.4f}, V0: {case['Heston_nu']:.4f}")

# Preço da PINN
price_pinn = case['Price_Pred']

# Preço Monte Carlo (Simulação)
# Nota: 'r' pode não estar no df_results, verifique seu data_loader. Usando fixo 0.10 ou pegando do data_stats se disponível.
price_mc = heston_mc_simulation(
    S0=case['S'], K=case['K'], T=case['T'], r=0.10,
    kappa=case['Heston_kappa'], theta=case['Heston_theta'], 
    xi=case['Heston_xi'], rho=case['Heston_rho'], v0=case['Heston_nu']
)

print(f"\nResultados:")
print(f"Preço PINN (Instantâneo): R$ {price_pinn:.4f}")
print(f"Preço Monte Carlo (5k caminhos): R$ {price_mc:.4f}")
print(f"Diferença Relativa: {abs(price_pinn - price_mc)/price_mc * 100:.2f}%")

if abs(price_pinn - price_mc)/price_mc < 0.05:
    print("✅ SUCESSO: A PINN convergiu para a solução física correta.")
else:
    print("⚠️ ATENÇÃO: Divergência entre PINN e Física. Verifique se o Loss PDE convergiu no treino.")

--- Validação Física (Monte Carlo) ---
Parâmetros Inferidos pela LSTM para este cenário:
Kappa: 0.5999, Theta: 0.4245, Xi: 0.7473, Rho: -0.1782, V0: 0.5577

Resultados:
Preço PINN (Instantâneo): R$ 0.0618
Preço Monte Carlo (5k caminhos): R$ 3.6458
Diferença Relativa: 98.30%
⚠️ ATENÇÃO: Divergência entre PINN e Física. Verifique se o Loss PDE convergiu no treino.


## 7: Análise dos Parâmetros Latentes (Regime de Mercado)

In [7]:
# Plota a evolução da volatilidade e correlação inferidas
fig_latent = viz.plot_latent_vol_evolution()
if fig_latent: fig_latent.show()

# Histograma dos parâmetros
fig_params = px.histogram(df_results, x="Heston_rho", nbins=50, title="Distribuição da Correlação (Rho) Inferida")
fig_params.show()

[INFO] ✓ Plot salvo: d:\UERJ\Programacao_e_Codigos\PINN_Opcoes_BR\resultados\plots\4a_latent_vol_evolution.html
